# 10. Дообучение манипулятора OMY с помощью π0.5 и оценка в MuJoCo

Этот Notebook (интерактивный блокнот) продолжает [1.collect_data.ipynb](1.collect_data.ipynb) и [2.visualize_data-omy.ipynb](2.visualize_data-omy.ipynb) и выполняет: проверку демонстраций → перенос данных → дополнение квантилей → пятишаговую проверку обучения → полноценное дообучение → офлайн-сравнение действий → захват и перенос в симуляции.

Объект — сценарий **захвата и переноса одной чашки манипулятором ROBOTIS OMY (шесть осей и захват)**, сцена `asset/example_scene_y.xml`. Используется отдельное окружение **Python 3.12 / LeRobot 0.6.1 / PyTorch 2.7.1 CUDA 12.8**; разделы 1–9 этой главы продолжают работать в своих старых окружениях. Не обновляйте зависимости в старом окружении и не запускайте этот процесс со старым `train_model.py`.

| Вход/выход | Определение в этом процессе |
| --- | --- |
| Два изображения | Внешняя камера и камера на запястье, RGB 256×256 |
| `observation.state` | 6-мерная поза эффектора `[x,y,z,roll,pitch,yaw]`: позиция в метрах, ориентация в радианах |
| `action` | 7-мерная абсолютная целевая позиция суставов `[joint1,…,joint6,gripper]`: углы в радианах, захват 0 — открыт, 1 — закрыт |
| Задача | Языковая инструкция `task` для каждого кадра, берётся из индекса задач исходных данных |
| `obj_init` | Два начальных положения чашки и тарелки; используется только для воспроизведения и сброса симуляции |
| Частота | 20 Гц [20 управляющих команд в секунду], сохраняется частота сбора |

**Область применения:** в разделе 5 сценария с языком состояние шестимерное и представляет углы суставов, а объекты и интерфейс сброса другие, поэтому преобразования и оценку отсюда нельзя напрямую переносить туда. Здесь используются данные позы эффектора из раздела 1. Обработку заполнения действий (по умолчанию до 32 измерений) модель выполняет внутри; семимерные действия вручную расширять до 32 не нужно. Поза эффектора и целевые углы суставов — разные величины, поэтому явно задано `use_relative_actions=False`.

Общие предобученные веса π0.5 — лишь стартовая точка дообучения, но не политика, уже умеющая захватывать и переносить объект OMY. По умолчанию обучается только action expert, визуальный энкодер заморожен, размер батча 1, включён gradient checkpointing. Официальный полный пример обучения рассчитан на мощное оборудование; эти настройки не гарантируют обучение на 6 ГБ видеопамяти. Рекомендуется запускать на Linux-рабочей станции лаборатории и ориентироваться на реальный пиковый расход видеопамяти при пятишаговой проверке. Данные, кэш моделей и выход храните на диске большой ёмкости.


## 1. Создание отдельного окружения

Следующие команды выполняются **один раз в терминале этой главы**, затем в меню блокнота выберите ядро `OMY pi05 (Python 3.12)`. Сначала установите предкомпилированный пакет PyTorch [фреймворк тензорных вычислений], соответствующий драйверу видеокарты, затем зависимости этого процесса. Нужны системный `ffmpeg` [утилита кодирования видео] и графический рабочий стол; окно MuJoCo [физический симулятор] требует службы дисплея. На сервере без рабочего стола можно запустить сервер блокнота через `xvfb-run` [виртуальный дисплей].

```bash
conda create -n omy-pi05 python=3.12 -y
conda activate omy-pi05
python -m pip install torch==2.7.1 torchvision==0.22.1 --index-url https://download.pytorch.org/whl/cu128
python -m pip install -r requirements_pi05.txt
python -m ipykernel install --user --name omy-pi05 --display-name 'OMY pi05 (Python 3.12)'
jupyter lab .
```

Если на рабочей станции используется другой инструмент управления окружениями, его можно разместить в `/data/Data14TB/envs/`. Перед началом задайте рабочий каталог в терминале, например:

```bash
export OMY_PI05_ROOT=/data/Data14TB/robotics_shared/experiments/omy-pi05
export HF_HOME=/data/Data14TB/robotics_shared/cache/huggingface
```

Ресурсы сцены берутся из установки этой главы: распакуйте `asset/objaverse/plate_11.zip` в тот же каталог. Не меняйте исходный каталог данных сбора. При первой загрузке предобученных весов нужны сеть и достаточно места на диске; короткая проверка и полное обучение сохраняют полные модели и состояния обучения, и вместе это может занимать десятки ГБ.


In [ ]:
import os
import sys
import json
import shlex
import subprocess
from pathlib import Path
from datetime import datetime

CHAPTER = Path.cwd().resolve()
assert (CHAPTER / "pi05_omy_utils.py").is_file(), "Start Jupyter in this chapter directory"
SOURCE = CHAPTER / "demo_data"  # Newly collected chapter-1 data; not demo_data_language.
RUN_ROOT = Path(os.environ.get("OMY_PI05_ROOT", CHAPTER / "pi05_outputs")).resolve()
V3_ROOT = RUN_ROOT / "omy_v3"
REPO_ID = "local/omy-pi05-20hz"
RUN_ID = datetime.now().strftime("%Y%m%d-%H%M%S")
SMOKE_OUTPUT = RUN_ROOT / f"smoke-{RUN_ID}"
TRAIN_OUTPUT = RUN_ROOT / f"train-{RUN_ID}"
BASE_MODEL = os.environ.get("OMY_PI05_PRETRAINED", "lerobot/pi05_base")
SMOKE_STEPS = 5
CHUNK_SIZE = 50
N_ACTION_STEPS = 5
TRAIN_STEPS = 10000
BATCH_SIZE = 1

# Set True only after checking the current collector and replaying your demonstrations.
ACTION_LABELS_VERIFIED = False
RUN_SMOKE = False
RUN_TRAIN = False

RUN_ROOT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(RUN_ROOT / "hf_cache"))
from pi05_omy_utils import (check_version, read_v21, prepare_dataset,
                            verify_alignment, training_command, load_policy, rollout)
print("source:", SOURCE)
print("outputs:", RUN_ROOT)


In [ ]:
import importlib.metadata as metadata
import torch

check_version()
assert sys.version_info[:2] == (3, 12), sys.version
assert torch.__version__.split("+")[0] == "2.7.1", torch.__version__
assert torch.cuda.is_available(), "Select the CUDA workstation kernel"
assert torch.cuda.is_bf16_supported(), "This configuration requires BF16 support"
assert metadata.version("mujoco") == "3.1.6"
free, total = torch.cuda.mem_get_info()
print("Python:", sys.version.split()[0])
print("LeRobot:", metadata.version("lerobot"))
print("PyTorch / CUDA:", torch.__version__, torch.version.cuda)
print("device:", torch.cuda.get_device_name(0))
print(f"free / total VRAM: {free/2**30:.1f} / {total/2**30:.1f} GiB")
if free < 20 * 2**30:
    print("Low free VRAM: reduce workload or use the larger workstation before training")


## 2. Сбор, воспроизведение и подтверждение меток действий

Сначала запустите раздел 1 в старом окружении сбора, увеличив `NUM_DEMO`, соберите полные успешные демонстрации с разными положениями чашки, затем воспроизведите их разделом 2. Для проверки процесса рекомендуется накопить 30–50 эпизодов; нужное количество в итоге зависит от покрытия задач и результатов оценки.

**Старые входы сбора до 2026-10-02 сохраняли измеренные состояния суставов как метки действий. Конвертация формата не восстановит утраченные целевые команды.** Соберите данные заново через исправленный интерфейс `get_commanded_joint_action()` или убедитесь, что источник данных уже использует этот интерфейс. Репозиторный `demo_data_example` — исторический пример структуры и не является значением по умолчанию для обучения.

Приведённые ниже проверки выявляют отсутствующие файлы, неверные размерности, временные метки, диапазоны захвата, проблемы с изображениями и индексами эпизодов, **но не могут по массивам определить источник меток действий или смысл состояния**. Убедившись, что поза эффектора из раздела 1, фактически отправленные целевые углы суставов и воспроизведение корректны, установите в ячейке конфигурации `ACTION_LABELS_VERIFIED = True`.


In [ ]:
info, episodes = read_v21(SOURCE)
print({key: info[key] for key in ["codebase_version", "fps", "total_episodes", "total_frames"]})
print("tasks:", sorted({row["task"] for rows in episodes for row in rows}))
print("first state:", episodes[0][0]["observation.state"])
print("first action:", episodes[0][0]["action"])
print("action semantics must be verified from the collector and replay")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pi05_omy_utils import _image

first = episodes[0][0]
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, key in zip(axes, ["observation.image", "observation.wrist_image"]):
    ax.imshow(_image(first[key], SOURCE.resolve(), (256, 256, 3)))
    ax.set_title(key)
    ax.axis("off")
plt.show()
actions = np.stack([row["action"] for row in episodes[0]])
plt.figure(figsize=(10, 3))
plt.plot(np.arange(len(actions))/20, actions)
plt.xlabel("simulation time (s)")
plt.ylabel("joint target (rad) / normalized gripper")
plt.legend([f"joint{i}" for i in range(1, 7)] + ["gripper"], ncol=4)
plt.show()


## 3. Преобразование в отдельный набор LeRobot v3

Изображения v2.1 этой главы хранятся в Parquet [колоночный файл данных]. Вспомогательная программа с помощью нового официального интерфейса записи данных заново собирает v3 эпизод за эпизодом, сохраняя изображения, шестимерное состояние, семимерные действия, задачи, начальные положения и частоту 20 Гц, и только нормализует имена ключей камер и индексы.

Целевой каталог не должен существовать; при повторном запуске выбирайте новый каталог — исходные данные и уже готовые результаты конвертации автоматически не удаляются. Если результат конвертации этого процесса уже есть, можно пропустить ячейку конвертации и сразу читать `omy_manifest.json` для проверки.

При трёх и более эпизодах около 20% полных эпизодов случайно откладываются для оценки с фиксированным seed 42. Точные `q01/q99` [1%/99% квантили], средние и стандартные отклонения состояний и действий считаются только по обучающим эпизодам; отложенные эпизоды для нормализации не используются. Один-два демонстрационных примера подходят только для проверки интерфейса или отладки переобучения и не дают оценки обобщающей успешности.


In [ ]:
manifest = prepare_dataset(SOURCE, V3_ROOT, REPO_ID,
                           labels_verified=ACTION_LABELS_VERIFIED, split_seed=42)
print(json.dumps(manifest, indent=2, ensure_ascii=False))


In [ ]:
# This cell can also reopen an existing conversion produced by this notebook.
manifest = json.loads((V3_ROOT / "omy_manifest.json").read_text(encoding="utf-8"))
print(verify_alignment(V3_ROOT, chunk_size=CHUNK_SIZE))
stats = json.loads((V3_ROOT / "meta/stats.json").read_text(encoding="utf-8"))
for key in ["observation.state", "action"]:
    print(key, "q01:", stats[key]["q01"], "q99:", stats[key]["q99"])
print("train episodes:", manifest["train_episodes"])
print("held-out episodes:", manifest["heldout_episodes"])


## 4. Проверка доступа к модели и подготовка команд обучения

По умолчанию дообучение использует [lerobot/pi05_base](https://huggingface.co/lerobot/pi05_base). Если есть совместимый локальный каталог предобученных весов, перед запуском задайте `OMY_PI05_PRETRAINED` или измените `BASE_MODEL` в ячейке конфигурации, чтобы использовать веса с диска. Каталог должен содержать модель, конфигурацию LeRobot и конфигурации входного и выходного процессоров; исходный каталог OpenPi нужно сначала подготовить этими адаптерами. Для обработки входа также нужен токенизатор `google/paligemma-3b-pt-224`. Если сообщается об ограниченном доступе, сначала примите лицензию на странице модели, затем выполните в терминале `hf auth login` [вход в учётную запись платформы моделей]; не записывайте токен доступа в блокнот.

Обучение запускается через `pi05_omy_train.py` из этой главы, который вызывает официальную точку входа обучения и проверяет, что все предобученные параметры загружены успешно. Фиксированная версия официального загрузчика может при ошибке чтения весов вернуть модель со случайной инициализацией, поэтому здесь такая ситуация считается ошибкой и останавливает процесс; при загрузке для инференса используется та же проверка.

Ключи камер сопоставляются как `observation.images.image` и `observation.images.image2`, признаки модели берутся из преобразованных метаданных. Язык вместе с квантованным состоянием эффектора поступает в токенизатор. И обучение, и инференс используют официальные входной и выходной процессоры; нельзя просто загрузить веса и подать в модель сырые массивы.

`chunk_size=50` [предсказание блока из 50 шагов действий] сохраняет значение по умолчанию предобученной модели; `n_action_steps=5` [исполняются 5 шагов подряд] означает, что политика заново наблюдает примерно каждые 0.25 с симуляционного времени. Короткая проверка подтверждает только цепочку данных, прямой и обратный проход, сохранение и загрузку, а не умение захватывать объект.


In [ ]:
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer

# Only config/tokenizer files are requested here; weights load when training starts.
if Path(BASE_MODEL).is_dir():
    required = ["config.json", "model.safetensors", "policy_preprocessor.json", "policy_postprocessor.json"]
    assert all((Path(BASE_MODEL) / name).is_file() for name in required), "Incomplete local pretrained directory"
    model_config = json.loads((Path(BASE_MODEL) / "config.json").read_text(encoding="utf-8"))
else:
    model_config = json.loads(Path(hf_hub_download(BASE_MODEL, "config.json")).read_text(encoding="utf-8"))
assert model_config["type"] == "pi05", model_config.get("type")
tokenizer = AutoTokenizer.from_pretrained("google/paligemma-3b-pt-224")
print("model configuration and tokenizer access: OK")
smoke_cmd = training_command(V3_ROOT, SMOKE_OUTPUT, steps=SMOKE_STEPS,
    batch_size=BATCH_SIZE, chunk_size=CHUNK_SIZE, n_action_steps=N_ACTION_STEPS,
    pretrained=BASE_MODEL)
train_cmd = training_command(V3_ROOT, TRAIN_OUTPUT, steps=TRAIN_STEPS,
    batch_size=BATCH_SIZE, chunk_size=CHUNK_SIZE, n_action_steps=N_ACTION_STEPS,
    pretrained=BASE_MODEL)
print(f"{SMOKE_STEPS}-step check:\n", shlex.join(smoke_cmd))
print("full training:\n", shlex.join(train_cmd))


In [ ]:
def run_training(command, log_path):
    # Stream logs; interrupting the cell stops only the process started by this cell.
    with Path(log_path).open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, cwd=CHAPTER, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                log.write(line)
                log.flush()
                print(line, end="")
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
    if returncode:
        raise RuntimeError(f"Training failed ({returncode}); inspect {log_path}")

if RUN_SMOKE:
    run_training(smoke_cmd, RUN_ROOT / f"smoke-{RUN_ID}.log")
    smoke_checkpoint = SMOKE_OUTPUT / "checkpoints/last/pretrained_model"
    assert (smoke_checkpoint / "config.json").is_file(), smoke_checkpoint
else:
    print("Set RUN_SMOKE=True in the configuration cell when ready")


## 5. Полноценное дообучение и восстановление с контрольных точек

После успешного завершения пятишаговой проверки изучите в логе функцию потерь, норму градиента, пиковую видеопамять и последний сохранённый полный checkpoint. Установите `RUN_TRAIN = True`, заново сгенерируйте команду полного обучения и выполните ячейку ниже.

Начальные параметры: 10000 шагов, батч 1, сохранение каждые 2000 шагов. Если данных мало, сначала следите за переобучением; окончательно пригодность модели определяется успешностью захвата и переноса на отложенных сценах. Число шагов и размер батча не проверены как оптимальные.

После прерывания можно восстановить модель, оптимизатор и планировщик по сохранённой полной конфигурации обучения. Команда восстановления имеет вид:

```bash
python -m pi05_omy_train --config_path=/absolute/path/train-*/checkpoints/last/pretrained_model/train_config.json --resume=true
```

Замените placeholder-пути реальными файлами и не направляйте новый выходной каталог на уже существующие результаты. Модель, сохранённая короткой проверкой, годится только для проверки цепочки; ниже оценка по умолчанию читает каталог полного обучения.


In [ ]:
if RUN_TRAIN:
    # Rebuild to ensure the output directory is still new before launching.
    train_cmd = training_command(V3_ROOT, TRAIN_OUTPUT, steps=TRAIN_STEPS,
        batch_size=BATCH_SIZE, chunk_size=CHUNK_SIZE, n_action_steps=N_ACTION_STEPS,
        pretrained=BASE_MODEL)
    run_training(train_cmd, RUN_ROOT / f"train-{RUN_ID}.log")
else:
    print("Set RUN_TRAIN=True after the five-step check passes")


## 6. Загрузка финального checkpoint и сравнение действий на отложенных данных

Если обучение шло в терминале или блокнот был открыт заново, сначала измените `CHECKPOINT` на фактический каталог `pretrained_model`. При загрузке читаются сохранённые конфигурации нормализации, токенизации и обработки устройств; статистики заново не оцениваются.

Офлайн-сравнение касается первой команды: шесть целевых углов суставов и команда захвата. Единицы у них разные, поэтому ошибки сообщаются отдельно. Офлайн-ошибка не заменяет успешность в симуляции; здесь проверяется не более восьми кадров — чтобы сначала убедиться в правильности формы выхода и значений.


In [ ]:
CHECKPOINT = TRAIN_OUTPUT / "checkpoints/last/pretrained_model"
policy, preprocessor, postprocessor = load_policy(CHECKPOINT)
print("state:", policy.config.input_features["observation.state"])
print("action:", policy.config.output_features["action"])


In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset
import pyarrow.parquet as pq

assert manifest["heldout_episodes"], "Collect at least three demos for a held-out evaluation"
dataset = LeRobotDataset(REPO_ID, root=V3_ROOT,
    delta_timestamps={"action": [i / 20 for i in range(policy.config.chunk_size)]})
index_rows = []
for path in sorted((V3_ROOT / "data").rglob("*.parquet")):
    index_rows.extend(pq.read_table(path).select(["index", "episode_index", "frame_index", "obj_init"]).to_pylist())
eval_indices = [r["index"] for r in index_rows if r["episode_index"] in manifest["heldout_episodes"]]
chosen = np.linspace(0, len(eval_indices) - 1, min(8, len(eval_indices)), dtype=int)
errors = []
for offset in chosen:
    sample = dataset[eval_indices[offset]]
    observation = {key: sample[key] for key in ["observation.images.image", "observation.images.image2", "observation.state", "task"]}
    policy.reset()
    preprocessor.reset()
    postprocessor.reset()
    with torch.inference_mode():
        predicted = postprocessor(policy.predict_action_chunk(preprocessor(observation)))[0, 0].cpu().numpy()
    target = sample["action"][0].numpy()
    assert predicted.shape == (7,) and np.isfinite(predicted).all()
    errors.append(np.abs(predicted - target))
errors = np.stack(errors)
print("joint target MAE (rad):", errors[:, :6].mean(axis=0))
print("gripper MAE:", errors[:, 6].mean())


## 7. Замкнутая оценка в сцене OMY с одной чашкой

Обучающая среда не зарегистрирована как официальная среда оценки, поэтому при обучении официальная оценка в среде отключена, и используется сцена этой главы. Сначала по `obj_init` отложенного эпизода задаются положения чашки и тарелки, затем политика самостоятельно выполняет захват и перенос, а реальные результаты проверки фиксируются.

На каждом шаге берутся текущие два изображения и поза эффектора, предсказываются абсолютные целевые углы суставов и передаются через `action_type='joint_angle'`. Каждый управляющий цикл продвигает физическую симуляцию на 0.05 с; частота совпадает с демонстрациями 20 Гц. Время инференса может замедлять воспроизведение по настенным часам, но не ускоряет данные. Синхронность управления здесь не означает выполнения требований реального времени на физическом роботе.

Среда обрезает команды захвата и ограничивает скорость открытия-закрытия, поэтому сохраняются как сырое предсказание модели, так и фактически отправленные действия. Управление манипулятором сохраняет упрощённый интерфейс симуляции этой главы; результат предназначен для проверки политики в данной сцене. Максимум 600 шагов на эпизод, т. е. 30 секунд симуляционного времени. Видео, действия и успешность записываются в новый каталог оценки и не перезаписывают старые результаты.


In [ ]:
import cv2
assert os.environ.get("DISPLAY"), "Run on the Ubuntu desktop or launch Jupyter with xvfb-run"
from mujoco_env.y_env import SimpleEnv
EVAL_OUTPUT = RUN_ROOT / f"eval-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
EVAL_OUTPUT.mkdir(exist_ok=False)
EVAL_EPISODES = manifest["heldout_episodes"][:5]
assert EVAL_EPISODES, "No held-out episodes; collect more demonstrations"
records = []
env = SimpleEnv(xml_path=str(CHAPTER / "asset/example_scene_y.xml"),
                action_type="joint_angle", state_type="ee_pose", seed=42)
try:
    for episode in EVAL_EPISODES:
        initial_row = next(r for r in index_rows if r["episode_index"] == episode and r["frame_index"] == 0)
        initial = np.asarray(initial_row["obj_init"], dtype=np.float32)
        env.reset(seed=42 + episode)
        env.set_obj_pose(initial[:3], initial[3:])
        for _ in range(100):
            env.step_env()
        task = dataset[initial_row["index"]]["task"]
        torch.manual_seed(42 + episode)
        result = rollout(policy, preprocessor, postprocessor, env, task=task, fps=20, max_steps=600)
        frames = result.pop("frames")
        video_path = EVAL_OUTPUT / f"episode-{episode:03d}.mp4"
        height, width = frames[0].shape[:2]
        writer = cv2.VideoWriter(str(video_path), cv2.VideoWriter_fourcc(*"mp4v"), 20, (width, height))
        assert writer.isOpened(), "Video encoder could not open the output"
        try:
            for frame in frames:
                writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
        finally:
            writer.release()
        result["predicted_actions"] = [a.tolist() for a in result["predicted_actions"]]
        result["applied_actions"] = [a.tolist() for a in result["applied_actions"]]
        result.update({"episode": episode, "seed": 42 + episode, "task": task, "video": str(video_path), "checkpoint": str(CHECKPOINT)})
        (EVAL_OUTPUT / f"episode-{episode:03d}.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
        records.append(result)
        print("episode:", episode, "success:", result["success"], "steps:", result["steps"])
finally:
    env.env.close_viewer()

summary = {"evaluated_episodes": EVAL_EPISODES,
           "successes": sum(r["success"] for r in records), "episodes": len(records),
           "success_rate": sum(r["success"] for r in records) / len(records),
           "checkpoint": str(CHECKPOINT)}
(EVAL_OUTPUT / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(summary)


## Проверка и частые вопросы

- В отдельном окружении запустите `python -m unittest -v test_pi05_omy`. Проверка использует настоящие новые интерфейсы записи и чтения данных и покрывает точность изображений и действий, перемешивание строк, заполнение действий в конце эпизода, квантили обучающих эпизодов, сохранение исходных данных и порядок отправки команд с частотой 20 Гц.
- Целевой каталог уже существует: используйте новый `V3_ROOT` или пропустите ячейку конвертации и читайте существующий манифест; исходные демонстрации не удаляйте.
- Нет `q01/q99` [1%/99% квантили]: убедитесь, что используются данные, преобразованные этим процессом, и проверьте `meta/stats.json`; не выдавайте средние и стандартные отклонения за квантили.
- Не читаются изображения или модель: проверьте, полностью ли загружены данные, принято ли соглашение о доступе и выбрано ли отдельное ядро.
- Не хватает видеопамяти для обучения: сначала оставьте батч 1 и настройки заморозки, запустите пятишаговую проверку на мощной рабочей станции. Сокращение блока действий меняет временное покрытие, поэтому одновременно проверяйте конфигурацию блока действий и число шагов исполнения при оценке.
- Низкая офлайн-ошибка, но провал в симуляции: сначала воспроизведите отложенные демонстрации и проверьте определение состояния, порядок камер, абсолютные суставные действия, начальные положения и направление захвата; затем увеличьте покрытие демонстраций и обучение. Не заявляйте о воспроизведении захвата и переноса по снижению loss.

Основания для поддержки этого процесса: официальное [руководство по π0.5](https://huggingface.co/docs/lerobot/main/en/pi05), [зафиксированные зависимости 0.6.1](https://github.com/huggingface/lerobot/blob/v0.6.1/pyproject.toml), [конфигурация π0.5](https://github.com/huggingface/lerobot/blob/v0.6.1/src/lerobot/policies/pi05/configuration_pi05.py) и [официальная точка входа обучения](https://github.com/huggingface/lerobot/blob/v0.6.1/src/lerobot/scripts/lerobot_train.py). Процесс жёстко использует 0.6.1: команды из последней документации других версий не обязательно совместимы.

Запуск этого блокнота не загружает автоматически данные, модели или результаты экспериментов. В записи о воспроизведении укажите число демонстраций, разбиение на обучение и отложенные данные, настройки обучения, checkpoint и фактическую успешность.
